<a href="https://colab.research.google.com/github/kirtiiiiiii/Construction-Cost-prediction/blob/main/Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ucimlrepo xgboost lightgbm shap statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from statsmodels.stats.outliers_influence import variance_inflation_factor

import shap

In [ ]:
import pandas as pd

file_path = "/content/Residential-Building-Data-Set.xlsx"

xls = pd.ExcelFile(file_path)

print(xls.sheet_names)

FileNotFoundError: [Errno 2] No such file or directory: '/content/Residential-Building-Data-Set.xlsx'

# Data preprocessing

In [ ]:
df = pd.read_excel(
    file_path,
    sheet_name="Data",
    header=1
)

print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
descriptions = pd.read_excel(
    file_path,
    sheet_name="Descriptions",
    header=None
)

descriptions.head(30)

In [ ]:
new_columns = [
    "START_YEAR",
    "START_QUARTER",
    "COMPLETION_YEAR",
    "COMPLETION_QUARTER",

    "V-1",
    "V-2",
    "V-3",
    "V-4",
    "V-5",
    "V-6",
    "V-7",
    "V-8",
]

# Add the 19 economic variables for each of the 5 time lags
for lag in range(1, 6):
    for variable in range(11, 30):
        new_columns.append(f"V-{variable}_Lag{lag}")

# Add output variables
new_columns.extend([
    "V-9",
    "V-10"
])

print("Number of new column names:", len(new_columns))
print("Number of dataset columns:", len(df.columns))

In [ ]:
df.columns = new_columns

df.head()

In [ ]:
df.info()

In [ ]:
print("Total missing values:", df.isnull().sum().sum())

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
y = df["V-10"]

In [ ]:
X = df.drop(columns=["V-9", "V-10"])

In [ ]:
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
date_features = [
    "START_YEAR",
    "START_QUARTER",
    "COMPLETION_YEAR",
    "COMPLETION_QUARTER"
]

project_features = [
    "V-1",
    "V-2",
    "V-3",
    "V-4",
    "V-5",
    "V-6",
    "V-7",
    "V-8"
]

economic_features = [
    col for col in df.columns
    if "_Lag" in col
]

In [ ]:
print("Date features:", len(date_features))
print("Project-specific features:", len(project_features))
print("Economic features:", len(economic_features))

print("\nProject features:")
print(project_features)

print("\nFirst 10 economic features:")
print(economic_features[:10])

Feature Separation:
The dataset contains four project date variables, eight project-specific physical and financial variables (V-1 to V-8), and 95 lagged economic variables. The economic variables consist of 19 indicators (V-11 to V-29) measured at five different time lags. ***The target variable selected for this study is V-10, representing actual construction cost. V-9, actual sales price, was excluded from the input features because it is also an output variable.***

# Outlier Analysis

In [ ]:
df.describe()

In [ ]:
print("Actual Construction Cost (V-10):")
print(y.describe())

In [ ]:
df["Cost_per_m2"] = df["V-10"] / df["V-2"]

df[["V-2", "V-10", "Cost_per_m2"]].head()

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(x=df["Cost_per_m2"])

plt.title("Boxplot of Actual Construction Cost per m²")
plt.xlabel("Cost per m²")
plt.show()

In [ ]:
Q1 = df["Cost_per_m2"].quantile(0.25)
Q3 = df["Cost_per_m2"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["Cost_per_m2"] < lower_bound) |
    (df["Cost_per_m2"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of potential outlier projects:", len(outliers))
print("Percentage of outliers:", round(len(outliers) / len(df) * 100, 2), "%")

In [ ]:
outliers[[
    "V-1",
    "V-2",
    "V-4",
    "V-7",
    "V-10",
    "Cost_per_m2"
]].sort_values(
    "Cost_per_m2",
    ascending=False
).head(10)

Outlier Analysis: A boxplot and the IQR method were used to identify projects with unusually high or low actual construction cost per square metre. Several potential high-cost outliers were detected. These observations were retained because they may represent genuine variation among real construction projects rather than data-entry errors. Removing them without domain justification could reduce the model's ability to predict unusually expensive projects.

In [ ]:
print("Number of economic features:", len(economic_features))
print(economic_features[:25])

# Multicollinearity

In [ ]:
economic_corr = df[economic_features].corr().abs()

upper_triangle = economic_corr.where(
    np.triu(
        np.ones(economic_corr.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = []

for column in upper_triangle.columns:
    correlated_features = upper_triangle.index[
        upper_triangle[column] > 0.95
    ].tolist()

    for feature in correlated_features:
        high_corr_pairs.append(
            (
                feature,
                column,
                upper_triangle.loc[feature, column]
            )
        )

high_corr_df = pd.DataFrame(
    high_corr_pairs,
    columns=[
        "Feature 1",
        "Feature 2",
        "Correlation"
    ]
)

print("Number of feature pairs with correlation > 0.95:")
print(len(high_corr_df))

high_corr_df.sort_values(
    "Correlation",
    ascending=False
).head(20)

In [ ]:
plt.figure(figsize=(20, 16))

sns.heatmap(
    df[economic_features].corr(),
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap of Lagged Economic Variables")
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

economic_data = df[economic_features].astype(float)

vif_data = pd.DataFrame()

vif_data["Feature"] = economic_data.columns

vif_data["VIF"] = [
    variance_inflation_factor(
        economic_data.values,
        i
    )
    for i in range(economic_data.shape[1])
]

vif_data = vif_data.sort_values(
    "VIF",
    ascending=False
)

vif_data.head(20)

In [ ]:
economic_corr = df[economic_features].corr().abs()

upper_triangle = economic_corr.where(
    np.triu(
        np.ones(economic_corr.shape),
        k=1
    ).astype(bool)
)

features_to_drop = [
    column for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.95)
]

print("Economic features before reduction:", len(economic_features))
print("Highly correlated features to remove:", len(features_to_drop))

reduced_economic_features = [
    feature for feature in economic_features
    if feature not in features_to_drop
]

print("Economic features after reduction:", len(reduced_economic_features))

In [ ]:
print("Remaining economic features:")

for feature in reduced_economic_features:
    print(feature)

In [ ]:
print("First 20 dropped features:")
print(features_to_drop[:20])

In [ ]:
final_feature_columns = (
    date_features
    + project_features
    + reduced_economic_features
)

X_final = df[final_feature_columns].copy()

print("Final feature shape:", X_final.shape)
print("Number of final features:", X_final.shape[1])

In [ ]:
reduced_economic_data = df[
    reduced_economic_features
].astype(float)

vif_after = pd.DataFrame()

vif_after["Feature"] = reduced_economic_data.columns

vif_after["VIF"] = [
    variance_inflation_factor(
        reduced_economic_data.values,
        i
    )
    for i in range(reduced_economic_data.shape[1])
]

vif_after = vif_after.sort_values(
    "VIF",
    ascending=False
)

vif_after.head(20)

Multicollinearity Analysis and Handling

The 95 lagged economic variables were examined using pairwise correlation and Variance Inflation Factor (VIF). The initial analysis revealed severe multicollinearity, with 992 feature pairs having an absolute correlation greater than 0.95. Several variables also produced infinite VIF values, indicating that they could be almost perfectly explained by combinations of other economic variables.

This result is expected because the same economic indicators were measured across five consecutive time lags, and many economic variables also change together over time. To reduce redundancy, one feature from each highly correlated pair was removed using an absolute correlation threshold of 0.95. The VIF analysis was then repeated on the reduced set of economic features.

In [ ]:
def calculate_vif(data):
    vif = pd.DataFrame()
    vif["Feature"] = data.columns
    vif["VIF"] = [
        variance_inflation_factor(data.values, i)
        for i in range(data.shape[1])
    ]
    return vif


def reduce_vif(data, threshold=10):

    data = data.copy().astype(float)
    removed_features = []

    while True:

        vif_data = calculate_vif(data)

        max_vif = vif_data["VIF"].max()

        if max_vif <= threshold:
            break

        feature_to_remove = vif_data.loc[
            vif_data["VIF"].idxmax(),
            "Feature"
        ]

        removed_features.append(feature_to_remove)
        data = data.drop(columns=[feature_to_remove])

    return data, removed_features, calculate_vif(data)


vif_reduced_economic_data, vif_removed_features, final_vif = reduce_vif(
    df[reduced_economic_features],
    threshold=10
)

print("Economic features before VIF reduction:", len(reduced_economic_features))
print("Features removed using VIF:", len(vif_removed_features))
print("Economic features remaining:", vif_reduced_economic_data.shape[1])

print("\nRemoved features:")
print(vif_removed_features)

In [ ]:
final_vif.sort_values(
    "VIF",
    ascending=False
)

In [ ]:
final_economic_features = vif_reduced_economic_data.columns.tolist()

print("Final economic features:")
print(final_economic_features)

print("\nNumber of final economic features:")
print(len(final_economic_features))

In [ ]:
final_feature_columns = (
    date_features
    + project_features
    + final_economic_features
)

X_final = df[final_feature_columns].copy()
y_final = df["V-10"].copy()

print("Final X shape:", X_final.shape)
print("Final y shape:", y_final.shape)

After applying a pairwise correlation threshold of 0.95, the economic feature set was reduced from 95 to 38 variables. However, several remaining variables still had very high VIF values, indicating multicollinearity arising from combinations of multiple predictors rather than only highly correlated pairs. Therefore, an iterative VIF-based procedure was applied, repeatedly removing the predictor with the highest VIF until all remaining economic variables had VIF values of 10 or below.

* 95 economic features
* Reduced to 38 using correlation > 0.95
* Reduced further to 9 using iterative VIF ≤ 10



**Final model data has:
372 projects,
21 input features,
1 target variable (v-10 actual construction cost)**

# Train test splitting (80/20)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=0.20,
    random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

In [ ]:
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

def evaluate_model(model, X_train, X_test, y_train, y_test):

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)

    # Calculate evaluation metrics
    rmse = np.sqrt(
        mean_squared_error(y_test, predictions)
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    return rmse, mae, r2, predictions

# Random forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

rf_rmse, rf_mae, rf_r2, rf_predictions = evaluate_model(
    rf_model,
    X_train,
    X_test,
    y_train,
    y_test
)

print("Random Forest Results")
print("---------------------")
print("RMSE:", rf_rmse)
print("MAE:", rf_mae)
print("R²:", rf_r2)

# XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

xgb_rmse, xgb_mae, xgb_r2, xgb_predictions = evaluate_model(
    xgb_model,
    X_train,
    X_test,
    y_train,
    y_test
)

print("XGBoost Results")
print("----------------")
print("RMSE:", xgb_rmse)
print("MAE:", xgb_mae)
print("R²:", xgb_r2)

# LightGBM

In [ ]:
from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42,
    verbosity=-1
)

lgbm_rmse, lgbm_mae, lgbm_r2, lgbm_predictions = evaluate_model(
    lgbm_model,
    X_train,
    X_test,
    y_train,
    y_test
)

print("LightGBM Results")
print("-----------------")
print("RMSE:", lgbm_rmse)
print("MAE:", lgbm_mae)
print("R²:", lgbm_r2)

# SVR

In [ ]:
from sklearn.svm import SVR

svr_model = SVR(
    kernel="rbf",
    C=100,
    epsilon=0.1
)

svr_rmse, svr_mae, svr_r2, svr_predictions = evaluate_model(
    svr_model,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

print("SVR Results")
print("-----------")
print("RMSE:", svr_rmse)
print("MAE:", svr_mae)
print("R²:", svr_r2)

# Comparing all models

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "LightGBM",
        "SVR"
    ],
    "RMSE": [
        rf_rmse,
        xgb_rmse,
        lgbm_rmse,
        svr_rmse
    ],
    "MAE": [
        rf_mae,
        xgb_mae,
        lgbm_mae,
        svr_mae
    ],
    "R²": [
        rf_r2,
        xgb_r2,
        lgbm_r2,
        svr_r2
    ]
})

results = results.sort_values(
    "RMSE",
    ascending=True
).reset_index(drop=True)

results

LightGBM achieved the best overall performance on the held-out test set, with the lowest RMSE of 29.34 and the highest R² score of 0.9693. XGBoost achieved the lowest MAE of 17.64, indicating slightly lower average absolute prediction error. However, because LightGBM achieved superior RMSE and R² values, it was selected as the best overall model.

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    results["Model"],
    results["RMSE"]
)

plt.title("Model Comparison Based on RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    results["Model"],
    results["R²"]
)

plt.title("Model Comparison Based on R²")
plt.xlabel("Model")
plt.ylabel("R²")

plt.show()

# 2-fold cross validation

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

kf = KFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

In [ ]:
rf_cv_scores = cross_val_score(
    RandomForestRegressor(
        n_estimators=300,
        random_state=42
    ),
    X_final,
    y_final,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

rf_cv_rmse = -rf_cv_scores.mean()

print("Random Forest 2-Fold CV RMSE:", rf_cv_rmse)

In [ ]:
xgb_cv_scores = cross_val_score(
    XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ),
    X_final,
    y_final,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

xgb_cv_rmse = -xgb_cv_scores.mean()

print("XGBoost 2-Fold CV RMSE:", xgb_cv_rmse)

In [ ]:
lgbm_cv_scores = cross_val_score(
    LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        random_state=42,
        verbosity=-1
    ),
    X_final,
    y_final,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

lgbm_cv_rmse = -lgbm_cv_scores.mean()

print("LightGBM 2-Fold CV RMSE:", lgbm_cv_rmse)

NameError: name 'X_final' is not defined

In [ ]:
svr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(
        kernel="rbf",
        C=100,
        epsilon=0.1
    ))
])

svr_cv_scores = cross_val_score(
    svr_pipeline,
    X_final,
    y_final,
    cv=kf,
    scoring="neg_root_mean_squared_error"
)

svr_cv_rmse = -svr_cv_scores.mean()

print("SVR 2-Fold CV RMSE:", svr_cv_rmse)

In [ ]:
cv_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "LightGBM",
        "SVR"
    ],
    "2-Fold CV RMSE": [
        rf_cv_rmse,
        xgb_cv_rmse,
        lgbm_cv_rmse,
        svr_cv_rmse
    ]
})

cv_results = cv_results.sort_values(
    "2-Fold CV RMSE"
).reset_index(drop=True)

cv_results

Although LightGBM achieved the best performance on the single 80/20 test split, with the lowest RMSE and highest R² score, XGBoost achieved the lowest RMSE during 2-fold cross-validation. In addition, XGBoost produced the lowest MAE on the test set. Therefore, **XGBoost was selected as the final model** because it demonstrated a better balance between predictive accuracy and performance stability across different data splits.

**FINAL MODEL: XGBOOST**

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_final.columns,
    "Importance": xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

feature_importance

In [ ]:
top_features = feature_importance.head(15)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.title("Top 15 Feature Importances from XGBoost")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.gca().invert_yaxis()

plt.show()

In [ ]:
feature_names = {
    "START_YEAR": "Project Start Year",
    "START_QUARTER": "Project Start Quarter",
    "COMPLETION_YEAR": "Project Completion Year",
    "COMPLETION_QUARTER": "Project Completion Quarter",

    "V-1": "Project Locality",
    "V-2": "Total Floor Area",
    "V-3": "Lot Area",
    "V-4": "Preliminary Estimated Construction Cost",
    "V-5": "Initial Estimated Construction Cost",
    "V-6": "Equivalent Preliminary Construction Cost",
    "V-7": "Construction Duration",
    "V-8": "Unit Price at Project Start",

    "V-11_Lag5": "Building Permits (Lag 5)",
    "V-19_Lag2": "Amount of Bank Loans (Lag 2)",
    "V-19_Lag4": "Amount of Bank Loans (Lag 4)",
    "V-19_Lag5": "Amount of Bank Loans (Lag 5)",
    "V-27_Lag1": "Stock Market Index (Lag 1)",
    "V-28_Lag1": "City Population (Lag 1)",
    "V-28_Lag2": "City Population (Lag 2)",
    "V-28_Lag3": "City Population (Lag 3)",
    "V-28_Lag4": "City Population (Lag 4)"
}

feature_importance["Feature Name"] = (
    feature_importance["Feature"]
    .map(feature_names)
    .fillna(feature_importance["Feature"])
)

feature_importance.head(15)

# SHAP analysis

In [ ]:
import shap

In [ ]:
explainer = shap.TreeExplainer(xgb_model)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values,
    X_test,
    feature_names=X_final.columns
)

Feature Importance and SHAP Analysis

Feature importance analysis using the XGBoost model showed that V-5, representing the initial estimated construction cost, was by far the most influential predictor, with an importance score of approximately 0.907. This indicates that preliminary cost estimates provide the strongest information for predicting the final actual construction cost.

The SHAP summary plot confirmed this result. Higher values of V-5 generally produced positive SHAP values, increasing the predicted construction cost, while lower values produced negative SHAP values. Construction duration (V-7), project completion year, equivalent preliminary construction cost (V-6), and lagged amounts of bank loans (V-19) also contributed to the predictions.

Overall, the results suggest that project-level preliminary cost estimates are more influential than the retained economic indicators, while financing conditions and project timing provide additional predictive information.

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    xgb_predictions
)

min_value = min(y_test.min(), xgb_predictions.min())
max_value = max(y_test.max(), xgb_predictions.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    "r--"
)

plt.xlabel("Actual Construction Cost")
plt.ylabel("Predicted Construction Cost")
plt.title("XGBoost: Actual vs Predicted Construction Cost")

plt.show()